# Heart Disease Prediction

Binary classification of heart disease on the **UCI Heart Disease** dataset (920 patients, 4 sources) using five models: Logistic Regression, Random Forest, KNN, XGBoost and a small Deep Neural Network.

**Pipeline:** load → preprocess → clean → feature importance → 80/20 split → standardise → train/evaluate → compare.

> Run from the repo root or from `notebooks/`. Requires `pip install -r requirements.txt`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

SEED = 42
DATA_DIR = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

## 1. Load dataset

In [ ]:
uci = pd.read_csv(DATA_DIR / "UCIdataset.csv")
print("UCI dataset shape:", uci.shape)
uci.head()

## 2. Preprocessing & cleaning

* Remove identifier columns (`id`, `dataset`).
* Drop the categorical columns (`sex`, `cp`, `restecg`, `slope`, `thal`) to keep the feature set purely numeric.
* Replace `'?'` with `NaN` and fill remaining missing values with the column mean.
* Binarise the target: `num > 0` → heart disease (1), otherwise 0.

In [ ]:
uci = uci.drop(columns=["id", "dataset", "sex", "cp", "restecg", "slope", "thal"])
uci = uci.replace("?", np.nan).apply(pd.to_numeric, errors="coerce")
uci = uci.fillna(uci.mean())

X = uci.drop(columns="num")
y = (uci["num"] > 0).astype(int)

print("Missing values after cleaning:", int(X.isnull().sum().sum()))
print("Features:", list(X.columns))
print("Class balance:\n", y.value_counts().rename({0: "No disease", 1: "Disease"}))

## 3. Feature importance

In [ ]:
temp_rf = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X, y)
feature_importance = pd.Series(temp_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(feature_importance)

feature_importance.sort_values().plot(kind="barh", title="Random Forest Feature Importance")
plt.show()

## 4. Train/test split and standardisation
Scaled data is used for Logistic Regression, KNN and the neural network; tree models use the raw features.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

accuracy_results, roc_results = {}, {}


def report(name, y_pred, y_prob, acc=None):
    """Print metrics for a model and store accuracy / ROC-AUC for the final comparison."""
    acc = accuracy_score(y_test, y_pred) if acc is None else acc
    auc = roc_auc_score(y_test, y_prob)
    print(f"===== {name.upper()} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification report:\n", classification_report(y_test, y_pred))
    accuracy_results[name], roc_results[name] = acc, auc

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax[0])
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[1], name=name)
    ax[0].set_title(f"{name} - Confusion Matrix")
    ax[1].set_title(f"{name} - ROC Curve")
    plt.tight_layout()
    plt.show()

## 5. Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
report("Logistic Regression", log_model.predict(X_test_scaled), log_model.predict_proba(X_test_scaled)[:, 1])

## 6. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_train, y_train)
report("Random Forest", rf.predict(X_test), rf.predict_proba(X_test)[:, 1])

## 7. K-Nearest Neighbours

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
report("KNN", knn.predict(X_test_scaled), knn.predict_proba(X_test_scaled)[:, 1])

## 8. XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4,
    random_state=SEED, eval_metric="logloss",
).fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
report("XGBoost", y_pred_xgb, xgb.predict_proba(X_test)[:, 1])

## 9. Deep Neural Network
Two hidden layers (32 and 16 ReLU units) and a sigmoid output, trained for 50 epochs with a 20% validation split. Results vary slightly between runs even with a fixed seed, depending on hardware and TensorFlow version.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

tf.keras.utils.set_random_seed(SEED)

model_nn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid"),
])
model_nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model_nn.fit(X_train_scaled, y_train, epochs=50, batch_size=16, validation_split=0.2, verbose=1)

y_prob_nn = model_nn.predict(X_test_scaled).ravel()
y_pred_nn = (y_prob_nn > 0.5).astype(int)
report("Deep Neural Network", y_pred_nn, y_prob_nn)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 10. Model comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Model": list(accuracy_results),
    "Accuracy": list(accuracy_results.values()),
    "ROC-AUC": list(roc_results.values()),
})
print(comparison_df.round(4).to_string(index=False))

comparison_df.set_index("Model")[["Accuracy", "ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), title="Model comparison (UCI)"
)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 11. Predictions on the test set (XGBoost)

In [ ]:
labels = np.where(y_pred_xgb == 1, "Heart Disease Detected", "Heart Disease Not Detected")

results_df = X_test.copy()
results_df["Actual"] = y_test.map({0: "No disease", 1: "Disease"}).values
results_df["Predicted"] = labels
results_df[["Actual", "Predicted"]].head(20)

## Notes & limitations
* Categorical features (`sex`, `cp`, `restecg`, `slope`, `thal`) are dropped; encoding them is a natural next step.
* Several UCI rows have `chol = 0` (172 rows, mostly the Switzerland and VA sources), which really means "not recorded" but is currently treated as a real value.
* Mean imputation and the scaler are applied around a single 80/20 split; cross-validation would give more reliable estimates on a dataset of this size.